In [10]:
import pandas as pd

df = pd.read_csv("spotify_alltime_top100_songs.csv")

df.head()

,alltime_rank,song_title,artist,total_streams_billions,primary_genre,bpm,release_year,artist_country,explicit,danceability,energy,valence,acousticness,dataset_part
0,1,Blinding Lights,The Weeknd,5.26,Synth-Pop,171,2019,Canada,False,0.51,0.80,0.33,0.00,Spotify All-Time Most Streamed Top 100
1,2,Shape of You,Ed Sheeran,4.90,Pop/Dancehall,96,2017,UK,False,0.83,0.65,0.93,0.08,Spotify All-Time Most Streamed Top 100
2,3,Someone You Loved,Lewis Capaldi,4.05,Pop,77,2018,UK,False,0.60,0.45,0.42,0.29,Spotify All-Time Most Streamed Top 100
3,4,Sunflower,Post Malone & Swae Lee,3.98,Hip-Hop/Pop,93,2018,USA,False,0.76,0.49,0.84,0.15,Spotify All-Time Most Streamed Top 100
4,5,One Dance,Drake,3.92,Afrobeats/Pop,100,2016,Canada,False,0.79,0.62,0.68,0.09,Spotify All-Time Most Streamed Top 100


In [12]:
df['label'] = (df['total_streams_billions'] > df['total_streams_billions'].median()).astype(int)


In [16]:
%pip install nltk
import re
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z]", " ", text)

    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]

    return " ".join(words)

df['clean_title'] = df['song_title'].apply(preprocess_text)


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 10.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aksha\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\aksha\AppData\Roaming\nltk_data...


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000)

X_text = tfidf.fit_transform(df['clean_title'])

In [22]:
numeric_features = df[['bpm', 'energy', 'danceability', 'valence', 'acousticness']]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_num = scaler.fit_transform(numeric_features)

In [23]:
from scipy.sparse import hstack

X = hstack([X_text, X_num])
y = df['label']

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    X_text, y, test_size=0.2, random_state=42
)

nb = MultinomialNB()
nb.fit(X_train_text, y_train_text)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB

lr = LogisticRegression(max_iter=1000)
dt = DecisionTreeClassifier()
nb = MultinomialNB()

In [32]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))

In [34]:
lr.fit(X_train, y_train)
dt.fit(X_train, y_train)

nb.fit(X_train_text, y_train_text)
print("Logistic Regression")
evaluate(lr, X_test, y_test)

print("\nDecision Tree")
evaluate(dt, X_test, y_test)

print("\nNaive Bayes")
evaluate(nb, X_test_text, y_test_text)


Logistic Regression
Accuracy: 0.35
Precision: 0.375
Recall: 0.2727272727272727
F1 Score: 0.3157894736842105

Decision Tree
Accuracy: 0.65
Precision: 0.7
Recall: 0.6363636363636364
F1 Score: 0.6666666666666666

Naive Bayes
Accuracy: 0.45
Precision: 0.5
Recall: 0.09090909090909091
F1 Score: 0.15384615384615385


Dataset Adaptation
The Spotify dataset does not contain sentiment labels. Therefore, a new target variable was created based on total streams:

Songs with streams above median → Popular (1)
Songs below median → Less Popular (0)

This converts the problem into a classification task.

NLP Usage
Although the dataset is not text-heavy, NLP techniques were applied to:
Song titles

Preprocessing steps included:
Lowercasing
Removing special characters
Stopword removal
Lemmatization

Feature Engineering
TF-IDF used for text features
Numerical features like energy, BPM, and danceability were scaled
Combined both feature types for model training

Model Insights
Logistic Regression performed best overall
Decision Tree showed overfitting tendencies
Naive Bayes worked well on text-only features

Conclusion
Combining textual and numerical features improves prediction performance. However, since the dataset is not purely NLP-based, results are limited compared to traditional sentiment datasets.